# Imports


In [1]:
from __future__ import annotations

import math
import pickle
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, random_split

from IPython.display import HTML
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np

# Constants


In [2]:
RANDOM_SEED = 42

In [3]:
DATA_DIR: Path = Path("/home/linkezio/Datasets/cifar-10-python/cifar-10-batches-py")

In [4]:
MODELS_DIR: Path = Path("/home/linkezio/Projects/Efficient-Polling-Based-Learning-Rate-Optimization-for-Neural-Networks/models")

# Configs


## Seeds


In [5]:
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Device


In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


# Data


## Dataset Class


In [7]:
# Loads one CIFAR batch and returns images and labels.
def _load_cifar_batch(path: Path) -> tuple[np.ndarray, np.ndarray]:
    with path.open("rb") as f:
        obj = pickle.load(f, encoding="bytes")
    data = obj[b"data"]  # (N, 3072)
    labels = np.array(obj.get(b"labels") or obj.get(b"fine_labels"), dtype=np.int64)
    images = data.reshape(-1, 3, 32, 32)
    return images, labels


In [8]:
class CIFAR10Dataset(Dataset):
    def __init__(
        self,
        data_dir: Path,
        train: bool,
        mean: torch.Tensor | None = None,
        std: torch.Tensor | None = None,
    ):
        if train:
            batch_files = [data_dir / f"data_batch_{i}" for i in range(1, 6)]
        else:
            batch_files = [data_dir / "test_batch"]

        xs = []
        ys = []
        for p in batch_files:
            if not p.exists():
                raise FileNotFoundError(f"File not found: {p}")
            x, y = _load_cifar_batch(p)
            xs.append(x)
            ys.append(y)

        images = np.concatenate(xs, axis=0)
        labels = np.concatenate(ys, axis=0)

        self.images = torch.from_numpy(images).float()  # (N,3,32,32)
        self.labels = torch.from_numpy(labels).long()
        
        if (mean is None) ^ (std is None):
            raise ValueError("Pass `mean` and `std` together, or neither (data in [0, 1]).")
        self.mean = mean
        self.std = std

    # Returns the total number of dataset samples.
    def __len__(self) -> int:
        return int(self.labels.shape[0])

    # Returns one sample (x, y), with optional normalization.
    def __getitem__(self, idx: int):
        x = self.images[idx]
        if self.mean is not None:
            x = (x - self.mean) / self.std
        y = self.labels[idx]
        return x, y


## Calculate mean and std for normalize later


In [9]:

# Computes per-channel mean and standard deviation over the full dataset.
def compute_mean_std(dataset, batch_size=512):
    loader_mean = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    channel_sum = None
    n_pixels = 0
    for images, _ in loader_mean:
        b, c, h, w = images.shape
        if channel_sum is None:
            channel_sum = torch.zeros(c, dtype=torch.float64)
        channel_sum += images.double().sum(dim=(0, 2, 3))
        n_pixels += b * h * w

    mean = (channel_sum / n_pixels).float()

    loader_var = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    sum_sq = None
    for images, _ in loader_var:
        b, c, h, w = images.shape
        if sum_sq is None:
            sum_sq = torch.zeros(c, dtype=torch.float64)
        diff = images.double() - mean.view(1, c, 1, 1).double()
        sum_sq += (diff * diff).sum(dim=(0, 2, 3))

    var = (sum_sq / n_pixels).float()
    std = torch.sqrt(var)
    std = torch.clamp(std, min=1e-8)
    return mean, std


In [10]:
train_for_stats = CIFAR10Dataset(DATA_DIR, train=True)

cifar_mean, cifar_std = compute_mean_std(train_for_stats)

cifar_mean = cifar_mean.view(3, 1, 1)
cifar_std = cifar_std.view(3, 1, 1)

print("mean (R,G,B):", cifar_mean.squeeze().tolist())
print("std  (R,G,B):", cifar_std.squeeze().tolist())

/tmp/ipykernel_26046/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


mean (R,G,B): [125.30691528320312, 122.95039367675781, 113.86538696289062]
std  (R,G,B): [62.993221282958984, 62.088706970214844, 66.70490264892578]


## Train, Validation, Test Split


In [11]:
class DataLoaderHyperparameters:
    batch_size: int = 2048
    val_fraction: float = 0.1
    num_workers: int = 0 # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).

data_loader_hyperparameters = DataLoaderHyperparameters()

In [12]:
full_train = CIFAR10Dataset(DATA_DIR, train=True, mean=cifar_mean, std=cifar_std)
test_ds = CIFAR10Dataset(DATA_DIR, train=False, mean=cifar_mean, std=cifar_std)

val_size = max(1, int(len(full_train) * data_loader_hyperparameters.val_fraction))
train_size = len(full_train) - val_size

train_ds, val_ds = random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=True,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
val_loader = DataLoader(
    val_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
test_loader = DataLoader(
    test_ds,
    batch_size=data_loader_hyperparameters.batch_size,
    shuffle=False,
    num_workers=data_loader_hyperparameters.num_workers,
    pin_memory=(DEVICE == "cuda"),
)

len(train_ds), len(val_ds), len(test_ds)


/tmp/ipykernel_26046/3686208571.py:4: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f, encoding="bytes")


(45000, 5000, 10000)

# Model


## Hyperparameters


In [13]:
class ModelHyperparameters:
    batch_size: int = 128
    epochs: int = 150
    lr: float = 1e-5  # Sec. 5.4 — ordinary ANN with ADAM (Model 4) uses alpha = 0.01
    weight_decay: float = 5e-4
    # Jupyter: use 0 (workers cannot resolve classes defined in __main__ during pickling).
    num_workers: int = 0

model_hyperparameters = ModelHyperparameters()

## Model Class


In [14]:
class SimpleCIFAR10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 10),
        )

    # Runs the model forward pass.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Train


## Metric Functions


In [15]:
# Computes average batch accuracy.
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


In [16]:
loss_fn = nn.CrossEntropyLoss()

## Epoch Functions


In [17]:
@torch.inference_mode()
# Evaluates the model for one epoch and returns average loss/accuracy.
def eval_epoch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[float, float]:
    model.eval()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        losses.append(loss_fn(logits, y).item())
        accs.append(accuracy(logits, y))
    return float(np.mean(losses)), float(np.mean(accs))


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        accs.append(accuracy(logits.detach(), y))
    return float(np.mean(losses)), float(np.mean(accs))


import copy

# Polling Method + Adam (paper Sec. 3.6-3.8, Sec. 5.2 Model 2): one backward, n Adam steps with
# distinct learning rates; forward each candidate on the training minibatch; keep highest accuracy.
def train_epoch_polling(
    model: nn.Module,
    loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    candidate_lrs: tuple[float, ...],
) -> tuple[float, float, float]:
    model.train()
    losses: list[float] = []
    accs: list[float] = []
    chosen_lrs: list[float] = []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()

        best_acc = -1.0
        best_lr = candidate_lrs[0]
        best_snapshot: tuple[dict, dict] | None = None

        for lr in candidate_lrs:
            for pg in optim.param_groups:
                pg["lr"] = lr
            snap_m = copy.deepcopy(model.state_dict())
            snap_o = copy.deepcopy(optim.state_dict())
            optim.step()

            with torch.no_grad():
                logits_try = model(x)
                acc_try = accuracy(logits_try, y)

            model.load_state_dict(snap_m)
            optim.load_state_dict(snap_o)

            if acc_try > best_acc:
                best_acc = acc_try
                best_lr = lr
                best_snapshot = (snap_m, snap_o)

        assert best_snapshot is not None
        model.load_state_dict(best_snapshot[0])
        optim.load_state_dict(best_snapshot[1])
        for pg in optim.param_groups:
            pg["lr"] = best_lr
        optim.step()

        with torch.no_grad():
            logits_final = model(x)
            losses.append(loss_fn(logits_final, y).item())
            accs.append(accuracy(logits_final, y))
        chosen_lrs.append(best_lr)

    return (
        float(np.mean(losses)),
        float(np.mean(accs)),
        float(np.mean(chosen_lrs)),
    )


# Trains the model for one epoch and returns average loss/accuracy.
def train_epoch_new_method(model: nn.Module, loader: DataLoader, optim: torch.optim.Optimizer, loss_fn: nn.Module, lr_scheduler: None) -> tuple[float, float]:
    model.train()
    losses = []
    accs = []
    old_lr = optim.param_groups[0]["lr"]
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optim.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optim.step()

        losses.append(loss.item())
        acc = accuracy(logits.detach(), y)
        accs.append(acc)

        new_lr = lr_scheduler.step(optim.param_groups[0]["lr"], loss.item(), acc)
        for pg in optim.param_groups:
            pg["lr"] = new_lr

    #print(f"old, new LR: {old_lr}, {new_lr}")
    #print(f"LE difference: {new_lr - old_lr}")
    #print(f"patient count: {lr_scheduler.patient_counter}")

    return float(np.mean(losses)), float(np.mean(accs))


import copy




## Trainings

### Training X (Baseline)


In [18]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(float(optim.param_groups[0]["lr"]))

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [19]:
model = SimpleCIFAR10CNN().to(DEVICE)
optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [20]:
history, best_val_acc, model_path = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "cifar10_best.pt",
)

epoch 01/150 | train loss 2.3021 acc 0.0999 | val loss 2.3010 acc 0.0995
epoch 02/150 | train loss 2.2997 acc 0.1218 | val loss 2.2982 acc 0.1508
epoch 03/150 | train loss 2.2960 acc 0.1409 | val loss 2.2931 acc 0.1337
epoch 04/150 | train loss 2.2894 acc 0.1242 | val loss 2.2846 acc 0.1314
epoch 05/150 | train loss 2.2786 acc 0.1368 | val loss 2.2710 acc 0.1624
epoch 06/150 | train loss 2.2609 acc 0.1713 | val loss 2.2494 acc 0.1890
epoch 07/150 | train loss 2.2328 acc 0.1936 | val loss 2.2150 acc 0.1972
epoch 08/150 | train loss 2.1907 acc 0.2045 | val loss 2.1674 acc 0.2072
epoch 09/150 | train loss 2.1407 acc 0.2094 | val loss 2.1199 acc 0.2184
epoch 10/150 | train loss 2.1002 acc 0.2232 | val loss 2.0876 acc 0.2189
epoch 11/150 | train loss 2.0749 acc 0.2294 | val loss 2.0690 acc 0.2325
epoch 12/150 | train loss 2.0596 acc 0.2366 | val loss 2.0563 acc 0.2386
epoch 13/150 | train loss 2.0468 acc 0.2396 | val loss 2.0444 acc 0.2463
epoch 14/150 | train loss 2.0355 acc 0.2565 | val l

### Training Y (Library LR Scheduler)


In [21]:
class SchedulerHyperparameters:
    epochs: int = model_hyperparameters.epochs
    max_lr: float = model_hyperparameters.lr
    min_lr: float = 0.00000000000000000000000000000000001

scheduler_hyperparameters = SchedulerHyperparameters()


In [22]:
# Trains with a built-in PyTorch scheduler (CosineAnnealingWarmRestarts).
def fit_model_with_library_scheduler(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scheduler_hp: SchedulerHyperparameters,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optim,
        T_0=max(1, scheduler_hp.epochs // 4),
        T_mult=2,
        eta_min=scheduler_hp.min_lr,
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, scheduler_hp.epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optim, loss_fn)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        scheduler.step()
        current_lr = float(optim.param_groups[0]["lr"])

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(current_lr)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{scheduler_hp.epochs} | "
            f"lr {current_lr:.6f} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [23]:
model_scheduler = SimpleCIFAR10CNN().to(DEVICE)
optim_scheduler = torch.optim.Adam(
    model_scheduler.parameters(),
    lr=scheduler_hyperparameters.max_lr,
    weight_decay=model_hyperparameters.weight_decay,
)

In [24]:
history_scheduler, best_val_acc_scheduler, model_path_scheduler = fit_model_with_library_scheduler(
    model=model_scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim_scheduler,
    loss_fn=loss_fn,
    scheduler_hp=scheduler_hyperparameters,
    model_path=MODELS_DIR / "cifar10_best_scheduler.pt",
)


epoch 01/150 | lr 0.000010 | train loss 2.3029 acc 0.1185 | val loss 2.3014 acc 0.1114
epoch 02/150 | lr 0.000010 | train loss 2.3011 acc 0.1010 | val loss 2.2993 acc 0.1048
epoch 03/150 | lr 0.000010 | train loss 2.2981 acc 0.1048 | val loss 2.2953 acc 0.1211
epoch 04/150 | lr 0.000010 | train loss 2.2923 acc 0.1254 | val loss 2.2875 acc 0.1385
epoch 05/150 | lr 0.000010 | train loss 2.2811 acc 0.1435 | val loss 2.2728 acc 0.1587
epoch 06/150 | lr 0.000009 | train loss 2.2610 acc 0.1662 | val loss 2.2478 acc 0.1805
epoch 07/150 | lr 0.000009 | train loss 2.2307 acc 0.1898 | val loss 2.2143 acc 0.1947
epoch 08/150 | lr 0.000009 | train loss 2.1944 acc 0.1997 | val loss 2.1784 acc 0.2020
epoch 09/150 | lr 0.000009 | train loss 2.1589 acc 0.2099 | val loss 2.1452 acc 0.2126
epoch 10/150 | lr 0.000008 | train loss 2.1285 acc 0.2177 | val loss 2.1188 acc 0.2193
epoch 11/150 | lr 0.000008 | train loss 2.1064 acc 0.2267 | val loss 2.1005 acc 0.2293
epoch 12/150 | lr 0.000008 | train loss 2.0

### Training Z (Polling Method)


In [25]:
class PollingHyperparameters:
    candidate_lrs: tuple[float, ...] = (
        float(model_hyperparameters.lr * 0.25),
        float(model_hyperparameters.lr * 0.5),
        float(model_hyperparameters.lr),
        float(model_hyperparameters.lr * 2.0),
        float(model_hyperparameters.lr * 4.0),
    )


polling_hyperparameters = PollingHyperparameters()

In [26]:
def fit_model_polling(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    loss_fn: nn.Module,
    epochs: int,
    candidate_lrs: tuple[float, ...],
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history: dict[str, list[float]] = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, mean_lr = train_epoch_polling(
            model, train_loader, optim, loss_fn, candidate_lrs
        )
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(mean_lr)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"mean train-chosen lr {mean_lr:.6f} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path



In [27]:
model_polling = SimpleCIFAR10CNN().to(DEVICE)
optim_polling = torch.optim.Adam(
    model_polling.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)


In [28]:
history_polling, best_val_acc_polling, model_path_polling = fit_model_polling(
    model=model_polling,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim_polling,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    candidate_lrs=polling_hyperparameters.candidate_lrs,
    model_path=MODELS_DIR / "cifar10_best_polling.pt",
)


epoch 01/150 | mean train-chosen lr 0.000016 | train loss 2.3011 acc 0.1493 | val loss 2.3009 acc 0.1466
epoch 02/150 | mean train-chosen lr 0.000008 | train loss 2.2998 acc 0.1468 | val loss 2.2989 acc 0.1409
epoch 03/150 | mean train-chosen lr 0.000019 | train loss 2.2939 acc 0.1462 | val loss 2.2860 acc 0.1455
epoch 04/150 | mean train-chosen lr 0.000028 | train loss 2.2521 acc 0.1615 | val loss 2.2024 acc 0.1824
epoch 05/150 | mean train-chosen lr 0.000026 | train loss 2.1202 acc 0.2217 | val loss 2.0835 acc 0.2185
epoch 06/150 | mean train-chosen lr 0.000014 | train loss 2.0687 acc 0.2388 | val loss 2.0625 acc 0.2266
epoch 07/150 | mean train-chosen lr 0.000013 | train loss 2.0517 acc 0.2475 | val loss 2.0495 acc 0.2426
epoch 08/150 | mean train-chosen lr 0.000015 | train loss 2.0375 acc 0.2564 | val loss 2.0364 acc 0.2526
epoch 09/150 | mean train-chosen lr 0.000014 | train loss 2.0270 acc 0.2594 | val loss 2.0254 acc 0.2604
epoch 10/150 | mean train-chosen lr 0.000017 | train lo

### Training W (New Method)

In [ ]:
class NewLRScheduler:
    def __init__(
        self,
        min_lr=-float('inf'),
        max_lr=float('inf'),
        beta=0.5,
        patience=10
    ):
        self.min_lr = min_lr
        self.max_lr = max_lr
        self.beta = beta
        self.patience = patience

        self.ema = None
        self.prev_ema = float('inf')
        #
        self.first_loss = float('inf')
        self.last_loss = float('inf')
        self.patience_counter = 0
        self.increase_direction = True # Aumenta a Learning Rate

    # Ajusta a LR de acordo com a loss EMA
    def step(self, learning_rate, loss, accuracy):

        #self.calculate_EMA(loss)
        #delta = self.calculate_delta_loss()
        
        if self.patience_counter == 0:
            self.first_loss = loss
        
        if self.patience_counter == self.patience:
            # Salva a loss atual
            self.last_loss = loss
            # Se a loss for 0, não é necessário ajustar a LR
            if self.last_loss == 0:
                return learning_rate
            # Reseta o contador de paciência
            self.patience_counter = 0

            print(f"{self.first_loss / self.last_loss}")

            # Calcula a diferença entre a loss inicial e a loss atual
            loss_improvement = self.first_loss - self.last_loss
            # Se a loss melhorou ou estagnou, ajusta a LR
            if loss_improvement >= 0:
                if self.increase_direction:
                    # Ajusta a LR de acordo com a loss inicial e a loss atual
                    learning_rate *= 1 + math.log(self.first_loss / self.last_loss)
                else:
                    # Ajusta a LR de acordo com a loss inicial e a loss atual
                    learning_rate *= 1 + math.log(-(self.first_loss / self.last_loss) + 2)
            # Se a loss piorou, inverte a direção
            else:
                if self.increase_direction:
                    self.increase_direction = False
                else:
                    self.increase_direction = True

            print(f"LR: {learning_rate}, Direction: {self.increase_direction}, Patience: {self.patience_counter}")

        self.patience_counter += 1

        return min(self.max_lr, max(self.min_lr, learning_rate))

    # Calcula a EMA (Exponential Moving Average) da loss, é uma forma de suavizar a loss de acordo com as informações passadas, o parâmetro beta é o peso da informação passada (quanto mais próximo de 1, mais peso a informação passada tem)
    def calculate_EMA(self, loss):
        if self.ema is None:
            self.prev_ema = loss
            self.ema = loss
            return loss

        self.prev_ema = self.ema
        self.ema = self.beta * self.ema + (1 - self.beta) * loss

    # Calcula quantos porcento a loss EMA mudou (positivo = melhora; negativo = piora)
    def calculate_delta_loss(self):
        if self.prev_ema is None:
            return 0
        return (self.prev_ema - self.ema) / self.prev_ema

In [30]:
# Coordinates epoch training/validation and saves the best checkpoint.
def fit_model_new_scheduler(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optim: torch.optim.Optimizer,
    lr_scheduler: NewLRScheduler,
    loss_fn: nn.Module,
    epochs: int,
    model_path: Path,
) -> tuple[dict[str, list[float]], float, Path]:
    """Coordinates epoch training/validation and saves the best checkpoint."""
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_val_acc = -math.inf

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch_new_method(model, train_loader, optim, loss_fn, lr_scheduler)
        va_loss, va_acc = eval_epoch(model, val_loader, loss_fn)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["lr"].append(float(optim.param_groups[0]["lr"]))

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), model_path)

        print(
            f"epoch {epoch:02d}/{epochs} | "
            f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {va_loss:.4f} acc {va_acc:.4f}"
        )

    print("best val acc:", best_val_acc, "saved:", str(model_path))
    return history, best_val_acc, model_path


In [36]:
model = SimpleCIFAR10CNN().to(DEVICE)

optim = torch.optim.Adam(
    model.parameters(),
    lr=model_hyperparameters.lr,
    weight_decay=model_hyperparameters.weight_decay,
)

new_lr_scheduler = NewLRScheduler(min_lr=-float('inf'),
                                  max_lr=float('inf'),
                                  beta=0.5,
                                  patience=30
                                  )

In [37]:
history_new_scheduler, best_val_acc_new_scheduler, model_path_new_scheduler = fit_model_new_scheduler(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optim=optim,
    lr_scheduler=new_lr_scheduler,
    loss_fn=loss_fn,
    epochs=model_hyperparameters.epochs,
    model_path=MODELS_DIR / "cifar10_best_new_scheduler.pt",
)

epoch 01/150 | train loss 2.3023 acc 0.0997 | val loss 2.3017 acc 0.0997
1.0013763097693016
LR: 1.0013753635231302e-05, Direction: True, Patience: 0
epoch 02/150 | train loss 2.3004 acc 0.1200 | val loss 2.2994 acc 0.1472
1.0033885437767391
LR: 1.0047628317230597e-05, Direction: True, Patience: 0
epoch 03/150 | train loss 2.2972 acc 0.1319 | val loss 2.2953 acc 0.1151
epoch 04/150 | train loss 2.2913 acc 0.1231 | val loss 2.2876 acc 0.1302
1.0080455480351416
LR: 1.0128143531563717e-05, Direction: True, Patience: 0
epoch 05/150 | train loss 2.2803 acc 0.1487 | val loss 2.2734 acc 0.1554
1.020014643212892
LR: 1.032885278262872e-05, Direction: True, Patience: 0
epoch 06/150 | train loss 2.2604 acc 0.1716 | val loss 2.2493 acc 0.1696
1.0357007787311754
LR: 1.0691171157135256e-05, Direction: True, Patience: 0
epoch 07/150 | train loss 2.2297 acc 0.1919 | val loss 2.2142 acc 0.1974
epoch 08/150 | train loss 2.1887 acc 0.2053 | val loss 2.1703 acc 0.2019
1.0651223829707295
LR: 1.1365674003804

## Training animation


In [ ]:
# Animate training: compare one or more `history` dicts (train/val loss + optional lr).
# All histories must have the same epoch count. Columns follow list order (left → right).
#
# Example:
#   animate_training_compare(
#       [history_scheduler, history_polling],
#       titles=["Cosine warm restarts", "Polling + Adam"],
#   )

# Increase HTML animation budget (MB) to avoid dropped frames in long runs.
mpl.rcParams["animation.embed_limit"] = 80


def animate_training_compare(
    histories: list[dict],
    titles: list[str] | None = None,
    interval_ms: int = 80,
):
    if len(histories) < 1:
        raise ValueError("Pass a non-empty list of history dicts.")

    n = len(histories[0]["train_loss"])
    for i, h in enumerate(histories):
        if len(h["train_loss"]) != n:
            raise ValueError(
                f"All histories must have the same length (run 0 has {n}, run {i} has {len(h['train_loss'])})."
            )

    if titles is None:
        titles = [f"Run {j + 1}" for j in range(len(histories))]
    elif len(titles) != len(histories):
        raise ValueError("`titles` must have the same length as `histories`.")

    epochs = np.arange(1, n + 1)
    trains = [np.asarray(h["train_loss"], dtype=float) for h in histories]
    vals = [np.asarray(h["val_loss"], dtype=float) for h in histories]
    lrs = [np.asarray(h.get("lr", [float("nan")] * n), dtype=float) for h in histories]

    n_cols = len(histories)
    fig_w = max(12.0, 4.0 * n_cols)
    if n_cols == 1:
        fig, axes = plt.subplots(2, 1, figsize=(6.0, 7), constrained_layout=True)
        axes_loss = [axes[0]]
        axes_lr = [axes[1]]
    else:
        fig, axes = plt.subplots(2, n_cols, figsize=(fig_w, 7), constrained_layout=True)
        axes_loss = list(axes[0])
        axes_lr = list(axes[1])

    lines_train, lines_val = [], []
    dots_train, dots_val = [], []
    lines_lr, dots_lr = [], []
    vlines_loss, vlines_lr = [], []

    for j in range(n_cols):
        ax = axes_loss[j]
        ax.set_title(titles[j] + " — loss")
        ax.set_xlabel("epoch")
        ax.set_ylabel("loss")
        ax.grid(True, alpha=0.3)
        (lt,) = ax.plot([], [], label="train", color="C0")
        (lv,) = ax.plot([], [], label="val", color="C1")
        dt = ax.scatter([], [], color="C0", s=40, zorder=5)
        dv = ax.scatter([], [], color="C1", s=40, zorder=5)
        ax.legend(loc="upper right")
        lines_train.append(lt)
        lines_val.append(lv)
        dots_train.append(dt)
        dots_val.append(dv)
        vlines_loss.append(ax.axvline(1, color="gray", ls="--", alpha=0.5))

    lr_colors = [f"C{(k % 8) + 2}" for k in range(n_cols)]
    for j in range(n_cols):
        ax = axes_lr[j]
        ax.set_title(titles[j] + " — learning rate")
        ax.set_xlabel("epoch")
        ax.set_ylabel("lr")
        ax.grid(True, alpha=0.3)
        c = lr_colors[j]
        (ll,) = ax.plot([], [], color=c)
        dl = ax.scatter([], [], color=c, s=40, zorder=5)
        lines_lr.append(ll)
        dots_lr.append(dl)
        vlines_lr.append(ax.axvline(1, color="gray", ls="--", alpha=0.5))

    def init():
        for ax in axes_loss + axes_lr:
            ax.set_xlim(0.5, n + 0.5)
        y0 = float(min(*(float(t.min()) for t in trains), *(float(v.min()) for v in vals)))
        y1 = float(max(*(float(t.max()) for t in trains), *(float(v.max()) for v in vals)))
        pad = 0.05 * (y1 - y0 + 1e-9)
        for ax in axes_loss:
            ax.set_ylim(y0 - pad, y1 + pad)
        lr_stack = np.concatenate(lrs)
        if not np.isfinite(lr_stack).any():
            lr_lo, lr_hi = 0.0, 1.0
        else:
            lr_lo = float(np.nanmin(lr_stack))
            lr_hi = float(np.nanmax(lr_stack))
        lr_pad = 0.05 * (lr_hi - lr_lo + 1e-12)
        for ax in axes_lr:
            ax.set_ylim(lr_lo - lr_pad, lr_hi + lr_pad)
        return []

    def update(k: int):
        k = int(k)
        sl = slice(0, k + 1)
        ex = epochs[sl]

        for j in range(n_cols):
            lines_train[j].set_data(ex, trains[j][sl])
            lines_val[j].set_data(ex, vals[j][sl])
            dots_train[j].set_offsets(np.c_[ex[-1:], trains[j][sl][-1:]])
            dots_val[j].set_offsets(np.c_[ex[-1:], vals[j][sl][-1:]])
            lines_lr[j].set_data(ex, lrs[j][sl])
            dots_lr[j].set_offsets(np.c_[ex[-1:], lrs[j][sl][-1:]])

        xcur = float(epochs[k])
        for vl in vlines_loss + vlines_lr:
            vl.set_xdata([xcur, xcur])

        return []

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=n,
        init_func=init,
        interval=interval_ms,
        blit=False,
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())




In [ ]:
# Already returns IPython.display.HTML; do not wrap with HTML() again.
animate_training_compare(
    [history, history_scheduler, history_polling, history_new_scheduler],
    titles=["Baseline (fixed LR)", "CosineAnnealingWarmRestarts", "Polling + Adam", "New Method"],
)


# Test


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "cifar10_best.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")

test loss 2.2979 | test acc 0.1453


In [ ]:
model.load_state_dict(torch.load(MODELS_DIR / "cifar10_best_scheduler.pt", map_location=DEVICE))

test_loss, test_acc = eval_epoch(model, test_loader, loss_fn)

print(f"test loss {test_loss:.4f} | test acc {test_acc:.4f}")

test loss 2.2861 | test acc 0.1101
